# Phase 1c — Bipartite structural features (ablation C)

Phase 1 showed cross-timestep wallet-trajectory information lifts RF by +0.005 F1 / +0.017 PR-AUC. The hand-crafted trajectory features captured "wallet x history" but not "wallet x **position in the bipartite graph**". This notebook adds the missing piece: **graph-theoretic features computed on the causal cumulative bipartite (wallet, tx) graph** at each timestep.

## Three feature sets compared (all RF, no neural)

| Ablation | Features | Dim |
|---|---|---|
| **A (raw)** | per-tx 182 features | 182 |
| **B (raw + traj)** | + Phase 1 trajectory features | 285 |
| **C (raw + traj + L1)** | + bipartite structural features | 292 |

## Layer 1: causal bipartite structural features (7 per tx)

For each tx `T` at time `t`, compute on the **cumulative causal bipartite graph** at time `t` (all addr↔tx edges where the tx endpoint has `t' ≤ t`):

- `bp_pagerank` — `T`'s undirected PageRank in the cumulative bipartite graph at `t`
- `bp_in_degree`, `bp_out_degree` — incident input / output wallet counts
- `bp_max_neighbor_pr`, `bp_mean_neighbor_pr` — max / mean PageRank of `T`'s incident wallets
- `bp_max_input_neighbor_pr`, `bp_max_output_neighbor_pr` — same restricted to input / output side

These capture "is `T` connected to wallets that are themselves central in the bipartite flow network up to time `t`" — a signal that the 72 within-timestep tx-tx aggregate features in the raw 182 cannot express, because tx-tx edges in Elliptic are within-timestep only.

## Causality contract — identical to phase 1 / phase 2-simplified

For each tx `T` at time `t`, every input uses only data with `τ ≤ t`:
- Trajectory features: strict `τ < t` priors (via `np.searchsorted(..., side='left')`).
- Bipartite-structural features: edges with tx endpoint at `t' ≤ t`. `T`'s own edges are included (T exists at time t — that's information available to a real-time classifier), but no future-tx edges. PageRank is purely structural — does not use any tx labels.
- StandardScaler not used (RF doesn't need it).
- `wallets_features.txt` not loaded (lifetime aggregates would leak).
- AddrAddr edges not loaded (no timestamps).

## Data split — identical to phase 1 / phase 2-simplified

- Train: labelled txs with `t ≤ 34` (paper protocol)
- Test: labelled txs with `t ≥ 35`

## Why this is a defensible "statistics on networks" contribution

The Maganti (2026) result showed that within-timestep tx-tx graph structure is actively harmful under honest evaluation on Elliptic. **Bipartite addr↔tx structure is a different graph** that carries different information — specifically cross-timestep connectivity through wallet trajectories. If RF + L1 lifts over RF + traj, that's a clean finding: the bipartite cross-timestep view of the graph is informative in a way the within-timestep tx-tx view is not.


In [13]:
import time
import numpy as np
import pandas as pd
import scipy.sparse as sp
from pathlib import Path
from collections import defaultdict
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    f1_score, roc_auc_score, average_precision_score,
    classification_report, precision_recall_curve,
)

ROOT = Path.cwd()
while not (ROOT / "actors_data").exists() and ROOT.parent != ROOT:
    ROOT = ROOT.parent
WALLETS_DIR      = ROOT / "actors_data"
TRANSACTIONS_DIR = ROOT / "transactions_data"
assert WALLETS_DIR.exists() and TRANSACTIONS_DIR.exists()
print(f"repo root: {ROOT}")

# Same data split as phase 1 / phase 2-simplified
TRAIN_END    = 34
N_TIME_STEPS = 49
RANDOM_SEED  = 175
np.random.seed(RANDOM_SEED)

# Phase 1 trajectory engineering caps (unchanged)
MAX_INCIDENT_PER_SIDE = 32
MAX_CO_WALLETS        = 150
RECENCY_SENTINEL      = N_TIME_STEPS * 2
DECAY_RATE            = 0.2

# Layer 1 PageRank settings
PR_DAMP        = 0.85
PR_ITERATIONS  = 30


repo root: /Users/jarayliu/Documents/GitHub/stat175-final


In [14]:
print("Loading transactions...")
tx_features_df = pd.read_csv(TRANSACTIONS_DIR / "txs_features.csv")
tx_classes_df  = pd.read_csv(TRANSACTIONS_DIR / "txs_classes.csv")
tx_classes_df["label"] = tx_classes_df["class"].map({1: 1, 2: 0, 3: -1}).astype(np.int8)
tx_id_to_idx = {tid: i for i, tid in enumerate(tx_features_df["txId"].values)}
N_tx = len(tx_features_df)
tx_feat_cols = [c for c in tx_features_df.columns if c not in ("txId", "Time step")]
F_TX = len(tx_feat_cols)

merged = tx_features_df[["txId","Time step"]].merge(tx_classes_df[["txId","label"]], on="txId", how="left")
tx_time  = merged["Time step"].astype(np.int64).values
tx_label = merged["label"].fillna(-1).astype(np.int64).values
tx_X_raw = tx_features_df[tx_feat_cols].fillna(0).astype(np.float32).values
print(f"  N_tx={N_tx:,}  F_TX={F_TX}")
print(f"  labels: illicit={int((tx_label==1).sum()):,}  licit={int((tx_label==0).sum()):,}  unknown={int((tx_label==-1).sum()):,}")

print("\nLoading actor bipartite edges...")
addr_tx_df = pd.read_csv(WALLETS_DIR / "AddrTx_edgelist.txt")
tx_addr_df = pd.read_csv(WALLETS_DIR / "TxAddr_edgelist.txt")
wallet_addrs = sorted(set(addr_tx_df["input_address"].unique()) | set(tx_addr_df["output_address"].unique()))
wallet_addr_to_idx = {a: i for i, a in enumerate(wallet_addrs)}
N_w = len(wallet_addr_to_idx)
print(f"  unique wallets: {N_w:,}")

print("\nBuilding per-wallet timelines...")
def edges_with_time(edge_df, addr_col, tx_col):
    df = edge_df[[addr_col, tx_col]].copy()
    df["w"]      = df[addr_col].map(wallet_addr_to_idx)
    df["tx"]     = df[tx_col].map(tx_id_to_idx)
    df = df.dropna(subset=["w", "tx"])
    df["w"]      = df["w"].astype(np.int64)
    df["tx"]     = df["tx"].astype(np.int64)
    df["t"]      = tx_time[df["tx"].values]
    df["tx_lab"] = tx_label[df["tx"].values]
    return df[["w","tx","t","tx_lab"]]

at = edges_with_time(addr_tx_df, "input_address",  "txId")
ta = edges_with_time(tx_addr_df, "output_address", "txId")
incident_in  = defaultdict(list)
incident_out = defaultdict(list)
for tx, w in zip(at["tx"].values,  at["w"].values):  incident_in[int(tx)].append(int(w))
for tx, w in zip(ta["tx"].values,  ta["w"].values):  incident_out[int(tx)].append(int(w))

all_edges = pd.concat([at, ta], ignore_index=True)
all_edges = all_edges.drop_duplicates(subset=["w", "tx"]).sort_values(["w", "t"]).reset_index(drop=True)
g = all_edges.groupby("w", sort=False)
wallet_t_arr   = {}
wallet_tx_arr  = {}
wallet_lab_arr = {}
for w, sub in g:
    wallet_t_arr[int(w)]   = sub["t"].values.astype(np.int64)
    wallet_tx_arr[int(w)]  = sub["tx"].values.astype(np.int64)
    wallet_lab_arr[int(w)] = sub["tx_lab"].values.astype(np.int64)

wallet_has_illicit_by = {}
for w, labs in wallet_lab_arr.items():
    illicit_mask = (labs == 1)
    if illicit_mask.any():
        wallet_has_illicit_by[w] = int(wallet_t_arr[w][illicit_mask].min())

wallet_event_count = {w: len(tarr) for w, tarr in wallet_t_arr.items()}
print(f"  wallets with timelines: {len(wallet_t_arr):,}")
print(f"  total wallet-tx pairs: {len(all_edges):,}")
print(f"  wallets with any illicit history: {len(wallet_has_illicit_by):,}")


Loading transactions...
  N_tx=203,769  F_TX=182
  labels: illicit=4,545  licit=42,019  unknown=157,205

Loading actor bipartite edges...
  unique wallets: 822,942

Building per-wallet timelines...
  wallets with timelines: 822,942
  total wallet-tx pairs: 1,268,260
  wallets with any illicit history: 14,266


In [15]:
# Same as phase 2 simplified: produces traj_X (103 features per tx).
agg_feat_names = ["total_BTC", "fees", "num_input_addresses", "num_output_addresses"]
agg_feat_idxs  = [tx_feat_cols.index(c) for c in agg_feat_names]
total_btc_idx  = tx_feat_cols.index("total_BTC")
F_agg          = len(agg_feat_names)

def pick_top_wallets(wlist, k=MAX_INCIDENT_PER_SIDE):
    if len(wlist) <= k: return list(wlist)
    cnts  = np.array([wallet_event_count.get(w, 0) for w in wlist], dtype=np.int64)
    order = np.argsort(-cnts, kind="stable")
    return [wlist[i] for i in order[:k]]

def per_wallet_priors(w, t_query):
    tarr = wallet_t_arr.get(w)
    if tarr is None: return None
    cut = int(np.searchsorted(tarr, t_query, side="left"))
    if cut == 0: return None
    prior_t   = tarr[:cut]
    prior_lab = wallet_lab_arr[w][:cut]
    prior_tx  = wallet_tx_arr[w][:cut]
    illicit_mask = (prior_lab == 1)
    n_illicit = int(illicit_mask.sum())
    n_licit   = int((prior_lab == 0).sum())
    last_illicit_t = int(prior_t[illicit_mask].max()) if n_illicit > 0 else -1
    if n_illicit > 0:
        decay_w = np.exp(-DECAY_RATE * (t_query - prior_t[illicit_mask]).astype(np.float64))
        decayed_illicit_score = float(decay_w.sum())
    else:
        decayed_illicit_score = 0.0
    illicit_frac = n_illicit / max(n_illicit + n_licit, 1)
    co_wallets = set()
    for tx_i in prior_tx:
        tx_i_int = int(tx_i)
        for co_w in incident_in.get(tx_i_int,  []):
            if co_w != w: co_wallets.add(co_w)
        for co_w in incident_out.get(tx_i_int, []):
            if co_w != w: co_wallets.add(co_w)
        if len(co_wallets) >= MAX_CO_WALLETS: break
    n_co_wallets = len(co_wallets)
    n_co_illicit = sum(1 for cw in co_wallets
                       if wallet_has_illicit_by.get(cw, N_TIME_STEPS + 1) < t_query)
    unique_in_partners, unique_out_partners = set(), set()
    for tx_i in prior_tx:
        tx_i_int = int(tx_i)
        for co_w in incident_in.get(tx_i_int,  []):
            if co_w != w: unique_in_partners.add(co_w)
        for co_w in incident_out.get(tx_i_int, []):
            if co_w != w: unique_out_partners.add(co_w)
    n_in_partners  = len(unique_in_partners)
    n_out_partners = len(unique_out_partners)
    fan_asymmetry  = n_out_partners / max(n_in_partners, 1)
    age      = int(t_query - prior_t[0])
    recency  = int(t_query - prior_t[-1])
    n_recent_5 = int(((t_query - prior_t) <= 5).sum())
    n_recent_1 = int(((t_query - prior_t) <= 1).sum())
    if cut >= 2:
        iat = np.diff(prior_t.astype(np.float64))
        iat_mean = float(iat.mean()); iat_std = float(iat.std())
        burstiness = float((iat_std - iat_mean) / (iat_std + iat_mean + 1e-8))
    else:
        iat_mean, iat_std, burstiness = 0.0, 0.0, 0.0
    velocity = n_recent_5 / max(cut, 1)
    feat_vals = tx_X_raw[prior_tx][:, agg_feat_idxs]
    return {
        "n": int(cut), "n_illicit": n_illicit, "n_licit": n_licit,
        "illicit_frac": illicit_frac, "last_illicit_t": last_illicit_t,
        "decayed_illicit": decayed_illicit_score,
        "n_co_wallets": n_co_wallets, "n_co_illicit": n_co_illicit,
        "co_illicit_frac": n_co_illicit / max(n_co_wallets, 1),
        "n_in_partners": n_in_partners, "n_out_partners": n_out_partners,
        "fan_asymmetry": fan_asymmetry,
        "first_seen_t": int(prior_t[0]), "last_seen_t": int(prior_t[-1]),
        "age": age, "recency": recency,
        "n_recent_5": n_recent_5, "n_recent_1": n_recent_1,
        "iat_mean": iat_mean, "iat_std": iat_std,
        "burstiness": burstiness, "velocity": velocity,
        "feat_means": feat_vals.mean(axis=0), "feat_maxes": feat_vals.max(axis=0),
    }

def aggregate_side(summaries, side, t_T):
    n_total = len(summaries)
    valid   = [s for s in summaries if s is not None]
    n_w_prior = len(valid)
    out = {}; p = side
    out[f"{p}_n_wallets"]              = n_total
    out[f"{p}_n_wallets_with_prior"]   = n_w_prior
    out[f"{p}_frac_first_appearance"]  = (n_total - n_w_prior) / max(n_total, 1)
    if not valid:
        # NOTE: this branch's key order MUST match the non-early-return branch below,
        # so the resulting DataFrame's column order is consistent regardless of which
        # tx is processed first. Mismatched orderings here change RF feature
        # subsampling and create reproducibility drift vs phase 1.
        out[f"{p}_n_priors_sum"]                = 0.0
        out[f"{p}_n_priors_max"]                = 0.0
        # Group 1
        out[f"{p}_n_illicit_sum"]               = 0.0
        out[f"{p}_n_illicit_max"]               = 0.0
        out[f"{p}_n_licit_sum"]                 = 0.0
        out[f"{p}_n_wallets_with_illicit"]      = 0.0
        out[f"{p}_n_wallets_illicit_frac_gt0"]  = 0.0
        out[f"{p}_n_wallets_illicit_frac_gt50"] = 0.0
        out[f"{p}_illicit_frac_max"]            = 0.0
        out[f"{p}_illicit_frac_mean"]           = 0.0
        out[f"{p}_decayed_illicit_max"]         = 0.0
        out[f"{p}_decayed_illicit_sum"]         = 0.0
        out[f"{p}_recency_to_illicit_min"]      = float(RECENCY_SENTINEL)
        # Group 2
        out[f"{p}_co_illicit_sum"]              = 0.0
        out[f"{p}_co_illicit_max"]              = 0.0
        out[f"{p}_co_illicit_frac_max"]         = 0.0
        out[f"{p}_co_illicit_frac_mean"]        = 0.0
        out[f"{p}_n_co_wallets_sum"]            = 0.0
        # Group 3
        out[f"{p}_fan_asymmetry_max"]           = 0.0
        out[f"{p}_fan_asymmetry_mean"]          = 0.0
        out[f"{p}_n_in_partners_max"]           = 0.0
        out[f"{p}_n_out_partners_max"]          = 0.0
        out[f"{p}_frac_single_use"]             = 0.0
        # Group 4
        out[f"{p}_age_max"]                     = 0.0
        out[f"{p}_age_mean"]                    = 0.0
        out[f"{p}_recency_min"]                 = float(RECENCY_SENTINEL)
        out[f"{p}_n_recent_5_sum"]              = 0.0
        out[f"{p}_n_recent_5_max"]              = 0.0
        out[f"{p}_n_recent_1_sum"]              = 0.0
        out[f"{p}_velocity_max"]                = 0.0
        out[f"{p}_velocity_mean"]               = 0.0
        out[f"{p}_burstiness_max"]              = 0.0
        out[f"{p}_burstiness_mean"]             = 0.0
        out[f"{p}_iat_mean_min"]                = 0.0
        out[f"{p}_iat_std_max"]                 = 0.0
        # Prior tx feature aggregations
        for nm in agg_feat_names:
            out[f"{p}_prior_{nm}_mean_max"] = 0.0
            out[f"{p}_prior_{nm}_max_max"]  = 0.0
        return out
    ns        = np.array([s["n"]              for s in valid], dtype=np.float64)
    nis       = np.array([s["n_illicit"]      for s in valid], dtype=np.float64)
    nls       = np.array([s["n_licit"]        for s in valid], dtype=np.float64)
    ill_frac  = np.array([s["illicit_frac"]   for s in valid], dtype=np.float64)
    dec_ill   = np.array([s["decayed_illicit"]for s in valid], dtype=np.float64)
    last_ill  = np.array([s["last_illicit_t"] for s in valid], dtype=np.int64)
    has_ill   = (last_ill >= 0)
    rec_ill   = np.where(has_ill, t_T - last_ill, RECENCY_SENTINEL).astype(np.float64)
    co_ill   = np.array([s["n_co_illicit"]    for s in valid], dtype=np.float64)
    co_n     = np.array([s["n_co_wallets"]    for s in valid], dtype=np.float64)
    co_frac  = np.array([s["co_illicit_frac"] for s in valid], dtype=np.float64)
    fan_a    = np.array([s["fan_asymmetry"]   for s in valid], dtype=np.float64)
    n_inp    = np.array([s["n_in_partners"]   for s in valid], dtype=np.float64)
    n_outp   = np.array([s["n_out_partners"]  for s in valid], dtype=np.float64)
    ages = np.array([s["age"]      for s in valid], dtype=np.float64)
    recs = np.array([s["recency"]  for s in valid], dtype=np.float64)
    nr5  = np.array([s["n_recent_5"] for s in valid], dtype=np.float64)
    nr1  = np.array([s["n_recent_1"] for s in valid], dtype=np.float64)
    vel  = np.array([s["velocity"] for s in valid], dtype=np.float64)
    bur  = np.array([s["burstiness"] for s in valid], dtype=np.float64)
    iam  = np.array([s["iat_mean"] for s in valid], dtype=np.float64)
    ias  = np.array([s["iat_std"]  for s in valid], dtype=np.float64)
    feat_means = np.stack([s["feat_means"] for s in valid], axis=0)
    feat_maxes = np.stack([s["feat_maxes"] for s in valid], axis=0)
    out[f"{p}_n_priors_sum"] = float(ns.sum()); out[f"{p}_n_priors_max"] = float(ns.max())
    out[f"{p}_n_illicit_sum"] = float(nis.sum()); out[f"{p}_n_illicit_max"] = float(nis.max())
    out[f"{p}_n_licit_sum"] = float(nls.sum())
    out[f"{p}_n_wallets_with_illicit"] = float(has_ill.sum())
    out[f"{p}_n_wallets_illicit_frac_gt0"]  = float((ill_frac > 0.0).sum())
    out[f"{p}_n_wallets_illicit_frac_gt50"] = float((ill_frac > 0.5).sum())
    out[f"{p}_illicit_frac_max"] = float(ill_frac.max())
    out[f"{p}_illicit_frac_mean"] = float(ill_frac.mean())
    out[f"{p}_decayed_illicit_max"] = float(dec_ill.max())
    out[f"{p}_decayed_illicit_sum"] = float(dec_ill.sum())
    out[f"{p}_recency_to_illicit_min"] = float(rec_ill.min())
    out[f"{p}_co_illicit_sum"] = float(co_ill.sum()); out[f"{p}_co_illicit_max"] = float(co_ill.max())
    out[f"{p}_co_illicit_frac_max"] = float(co_frac.max())
    out[f"{p}_co_illicit_frac_mean"] = float(co_frac.mean())
    out[f"{p}_n_co_wallets_sum"] = float(co_n.sum())
    out[f"{p}_fan_asymmetry_max"] = float(fan_a.max())
    out[f"{p}_fan_asymmetry_mean"] = float(fan_a.mean())
    out[f"{p}_n_in_partners_max"] = float(n_inp.max())
    out[f"{p}_n_out_partners_max"] = float(n_outp.max())
    out[f"{p}_frac_single_use"] = sum(1 for s in valid if s["n"] == 1) / max(n_w_prior, 1)
    out[f"{p}_age_max"] = float(ages.max()); out[f"{p}_age_mean"] = float(ages.mean())
    out[f"{p}_recency_min"] = float(recs.min())
    out[f"{p}_n_recent_5_sum"] = float(nr5.sum()); out[f"{p}_n_recent_5_max"] = float(nr5.max())
    out[f"{p}_n_recent_1_sum"] = float(nr1.sum())
    out[f"{p}_velocity_max"] = float(vel.max()); out[f"{p}_velocity_mean"] = float(vel.mean())
    out[f"{p}_burstiness_max"] = float(bur.max()); out[f"{p}_burstiness_mean"] = float(bur.mean())
    out[f"{p}_iat_mean_min"] = float(iam.min()); out[f"{p}_iat_std_max"] = float(ias.max())
    for k, nm in enumerate(agg_feat_names):
        out[f"{p}_prior_{nm}_mean_max"] = float(feat_means[:, k].max())
        out[f"{p}_prior_{nm}_max_max"]  = float(feat_maxes[:, k].max())
    return out

print("Computing trajectory features for all txs (~30s)...")
t0 = time.time()
traj_rows = []
for tx_idx in range(N_tx):
    t_T = int(tx_time[tx_idx])
    in_w  = pick_top_wallets(incident_in.get(tx_idx,  []))
    out_w = pick_top_wallets(incident_out.get(tx_idx, []))
    in_summ  = [per_wallet_priors(w, t_T) for w in in_w]
    out_summ = [per_wallet_priors(w, t_T) for w in out_w]
    row = {}
    row.update(aggregate_side(in_summ,  "in",  t_T))
    row.update(aggregate_side(out_summ, "out", t_T))
    row["both_sides_have_illicit"] = float(
        row["in_n_wallets_with_illicit"] > 0 and row["out_n_wallets_with_illicit"] > 0)
    row["total_n_illicit_priors"]  = row["in_n_illicit_sum"] + row["out_n_illicit_sum"]
    row["total_n_wallets_with_illicit"] = row["in_n_wallets_with_illicit"] + row["out_n_wallets_with_illicit"]
    row["total_co_illicit"]        = row["in_co_illicit_sum"] + row["out_co_illicit_sum"]
    row["min_recency_to_illicit"]  = min(row["in_recency_to_illicit_min"], row["out_recency_to_illicit_min"])
    row["max_illicit_frac_either_side"] = max(row["in_illicit_frac_max"], row["out_illicit_frac_max"])
    row["max_decayed_illicit_either"]   = max(row["in_decayed_illicit_max"], row["out_decayed_illicit_max"])
    row["max_co_illicit_frac_either"]   = max(row["in_co_illicit_frac_max"], row["out_co_illicit_frac_max"])
    row["total_frac_first_appearance"]  = (
        (row["in_frac_first_appearance"] * max(row["in_n_wallets"], 1) +
         row["out_frac_first_appearance"] * max(row["out_n_wallets"], 1))
        / max(row["in_n_wallets"] + row["out_n_wallets"], 1))
    T_total_btc = float(tx_X_raw[tx_idx, total_btc_idx])
    max_prior_btc  = max(row["in_prior_total_BTC_max_max"],  row["out_prior_total_BTC_max_max"])
    mean_prior_btc = max(row["in_prior_total_BTC_mean_max"], row["out_prior_total_BTC_mean_max"])
    row["T_btc_vs_max_prior"]  = T_total_btc / max(max_prior_btc, 1.0)
    row["T_btc_vs_mean_prior"] = T_total_btc / max(mean_prior_btc, 1.0)
    traj_rows.append(row)
    if tx_idx > 0 and tx_idx % 50_000 == 0:
        print(f"  tx {tx_idx:>7,}/{N_tx:,}  ({time.time()-t0:.0f}s elapsed)")

traj_df = pd.DataFrame(traj_rows)
traj_X  = traj_df.values.astype(np.float32)
traj_cols = list(traj_df.columns)
F_TRAJ  = traj_X.shape[1]
print(f"  done: traj_X={traj_X.shape} ({time.time()-t0:.1f}s)")


Computing trajectory features for all txs (~30s)...
  tx  50,000/203,769  (6s elapsed)
  tx 100,000/203,769  (15s elapsed)
  tx 150,000/203,769  (23s elapsed)
  tx 200,000/203,769  (30s elapsed)
  done: traj_X=(203769, 103) (33.7s)


In [16]:
# =============================================================================
#  Layer 1: causal bipartite structural features (PageRank + degrees)
#
#  For each tx T at time t, compute on the cumulative bipartite graph at time t
#  (all addr<->tx edges where the tx endpoint has t' <= t):
#    - bp_pagerank             T's own PageRank in the cumulative bipartite graph
#    - bp_in_degree            number of input wallets
#    - bp_out_degree           number of output wallets
#    - bp_max_neighbor_pr      max PageRank among T's incident wallets
#    - bp_mean_neighbor_pr     mean PageRank among T's incident wallets
#    - bp_max_input_neighbor_pr   max PageRank among input wallets
#    - bp_max_output_neighbor_pr  max PageRank among output wallets
#
#  Causality: edges with tx.t > t are excluded. PageRank uses NO labels at all.
#  T's own edges are included (T is observed at time t — that's allowed).
# =============================================================================

print("Computing Layer 1 bipartite structural features...")

# Combined node space: 0..N_w-1 = wallets, N_w..N_w+N_tx-1 = txs
N_total = N_w + N_tx

# Master edge arrays: (wallet, tx, edge_active_from = tx_time)
W_edges = all_edges["w"].values.astype(np.int64)
T_edges = all_edges["tx"].values.astype(np.int64)
T_times = all_edges["t"].values.astype(np.int64)

# Pre-bucket edges by activation timestep, for fast incremental graph building
edge_order = np.argsort(T_times)
W_edges_sorted = W_edges[edge_order]
T_edges_sorted = T_edges[edge_order]
T_times_sorted = T_times[edge_order]

# Pre-compute for each tx its incident input / output wallets (de-duped)
incident_in_arr  = {tx: list(set(ws)) for tx, ws in incident_in.items()}
incident_out_arr = {tx: list(set(ws)) for tx, ws in incident_out.items()}

# Output arrays — one row per tx
bp_pagerank        = np.zeros(N_tx, dtype=np.float32)
bp_in_degree       = np.zeros(N_tx, dtype=np.float32)
bp_out_degree      = np.zeros(N_tx, dtype=np.float32)
bp_max_nbr_pr      = np.zeros(N_tx, dtype=np.float32)
bp_mean_nbr_pr     = np.zeros(N_tx, dtype=np.float32)
bp_max_in_nbr_pr   = np.zeros(N_tx, dtype=np.float32)
bp_max_out_nbr_pr  = np.zeros(N_tx, dtype=np.float32)

t0 = time.time()
for t in range(1, N_TIME_STEPS + 1):
    # All edges active by t: edges where tx_endpoint has time <= t
    cut = int(np.searchsorted(T_times_sorted, t, side="right"))
    if cut == 0:
        continue
    we = W_edges_sorted[:cut]
    te = T_edges_sorted[:cut]
    # Build symmetric (undirected) sparse adjacency over the combined node space
    rows = np.concatenate([we, N_w + te])
    cols = np.concatenate([N_w + te, we])
    data = np.ones(rows.shape[0], dtype=np.float32)
    A = sp.csr_matrix((data, (rows, cols)), shape=(N_total, N_total))

    # Degrees
    deg = np.asarray(A.sum(axis=1)).flatten()                         # [N_total]
    inv_deg = np.zeros_like(deg)
    nz = deg > 0
    inv_deg[nz] = 1.0 / deg[nz]

    # PageRank power iteration (undirected)
    teleport = (1.0 - PR_DAMP) / N_total
    pr = np.full(N_total, 1.0 / N_total, dtype=np.float32)
    for _ in range(PR_ITERATIONS):
        weighted = pr * inv_deg
        pr = teleport + PR_DAMP * (A @ weighted).astype(np.float32)
    # (Dangling-node redistribution is omitted — relative PR among connected nodes is
    #  what we use, and zero-degree nodes contribute nothing anyway.)

    # Extract features for txs at exactly t
    txs_at_t = np.where(tx_time == t)[0]
    for tx_idx in txs_at_t:
        in_w  = incident_in_arr.get(int(tx_idx),  [])
        out_w = incident_out_arr.get(int(tx_idx), [])
        all_nbr = list(set(in_w) | set(out_w))
        bp_pagerank[tx_idx]  = pr[N_w + tx_idx]
        bp_in_degree[tx_idx]  = float(len(in_w))
        bp_out_degree[tx_idx] = float(len(out_w))
        if all_nbr:
            nbr_pr = pr[all_nbr]
            bp_max_nbr_pr[tx_idx]  = float(nbr_pr.max())
            bp_mean_nbr_pr[tx_idx] = float(nbr_pr.mean())
        if in_w:
            bp_max_in_nbr_pr[tx_idx]  = float(pr[in_w].max())
        if out_w:
            bp_max_out_nbr_pr[tx_idx] = float(pr[out_w].max())

    if t % 5 == 0 or t == N_TIME_STEPS:
        print(f"  t={t:2d}  edges_active={cut:>9,}  txs_at_t={len(txs_at_t):>4,}  "
              f"({time.time()-t0:.1f}s elapsed)")

L1_X = np.column_stack([
    bp_pagerank, bp_in_degree, bp_out_degree,
    bp_max_nbr_pr, bp_mean_nbr_pr,
    bp_max_in_nbr_pr, bp_max_out_nbr_pr,
]).astype(np.float32)
L1_cols = [
    "bp_pagerank", "bp_in_degree", "bp_out_degree",
    "bp_max_neighbor_pr", "bp_mean_neighbor_pr",
    "bp_max_input_neighbor_pr", "bp_max_output_neighbor_pr",
]
F_L1 = L1_X.shape[1]
print(f"\n  done: L1_X={L1_X.shape} ({time.time()-t0:.1f}s)")

# Sanity: PageRank should be larger for higher-degree txs on average
print("\nSanity check (PageRank vs degree, on random labelled txs):")
rng = np.random.default_rng(0)
sample = rng.choice(np.where(tx_label != -1)[0], 5, replace=False)
for s in sample:
    print(f"  tx_idx={int(s):>6}  t={int(tx_time[s]):>2}  "
          f"in={int(bp_in_degree[s])}  out={int(bp_out_degree[s])}  "
          f"PR={bp_pagerank[s]:.2e}  max_nbr_PR={bp_max_nbr_pr[s]:.2e}")


Computing Layer 1 bipartite structural features...
  t= 5  edges_active=  189,571  txs_at_t=6,803  (0.6s elapsed)
  t=10  edges_active=  349,941  txs_at_t=6,727  (1.5s elapsed)
  t=15  edges_active=  450,202  txs_at_t=3,639  (2.4s elapsed)
  t=20  edges_active=  546,600  txs_at_t=4,291  (3.5s elapsed)
  t=25  edges_active=  695,089  txs_at_t=2,314  (4.8s elapsed)
  t=30  edges_active=  767,134  txs_at_t=2,483  (6.1s elapsed)
  t=35  edges_active=  873,958  txs_at_t=5,507  (7.5s elapsed)
  t=40  edges_active=1,012,737  txs_at_t=4,481  (9.0s elapsed)
  t=45  edges_active=1,179,046  txs_at_t=5,598  (10.4s elapsed)
  t=49  edges_active=1,268,260  txs_at_t=2,454  (11.6s elapsed)

  done: L1_X=(203769, 7) (11.6s)

Sanity check (PageRank vs degree, on random labelled txs):
  tx_idx=135460  t=34  in=38  out=2  PR=1.29e-05  max_nbr_PR=4.18e-06
  tx_idx=110145  t=25  in=1  out=1  PR=1.01e-06  max_nbr_PR=8.84e-07
  tx_idx= 58780  t=11  in=1  out=2  PR=1.21e-06  max_nbr_PR=1.69e-06
  tx_idx= 68334

In [17]:
# Same temporal split as phase 1 (train t<=34, test t>=35).
labelled   = (tx_label != -1)
train_mask = labelled & (tx_time <= TRAIN_END)
test_mask  = labelled & (tx_time >  TRAIN_END)
y_train = tx_label[train_mask]
y_test  = tx_label[test_mask]
test_t_arr = tx_time[test_mask]
print(f"train (t<= {TRAIN_END}): n={int(train_mask.sum()):,}  illicit_rate={y_train.mean():.4f}")
print(f"test  (t >  {TRAIN_END}): n={int(test_mask.sum()):,}  illicit_rate={y_test.mean():.4f}")

# Build the three feature sets
X_A_train = tx_X_raw[train_mask]
X_A_test  = tx_X_raw[test_mask]
X_B_train = np.concatenate([tx_X_raw[train_mask], traj_X[train_mask]], axis=1)
X_B_test  = np.concatenate([tx_X_raw[test_mask],  traj_X[test_mask]],  axis=1)
X_C_train = np.concatenate([X_B_train, L1_X[train_mask]], axis=1)
X_C_test  = np.concatenate([X_B_test,  L1_X[test_mask]],  axis=1)

print(f"\nFeature dims:")
print(f"  A (raw):              {X_A_train.shape[1]}")
print(f"  B (raw + traj):       {X_B_train.shape[1]}")
print(f"  C (raw + traj + L1):  {X_C_train.shape[1]}")

def fit_and_eval(X_train, X_test, name):
    t0 = time.time()
    rf = RandomForestClassifier(n_estimators=200, class_weight="balanced",
                                 n_jobs=-1, random_state=RANDOM_SEED)
    rf.fit(X_train, y_train)
    yp  = rf.predict(X_test)
    ypr = rf.predict_proba(X_test)[:, 1]
    f1   = f1_score(y_test, yp, pos_label=1, zero_division=0)
    auc  = roc_auc_score(y_test, ypr)
    prauc= average_precision_score(y_test, ypr)
    print(f"  [{name:25s}]  F1={f1:.4f}  AUC={auc:.4f}  PR-AUC={prauc:.4f}  ({time.time()-t0:.1f}s)")
    return rf, yp, ypr, f1, auc, prauc

print("\n=== Training RFs ===")
rf_A, yp_A, ypr_A, f1_A, auc_A, prauc_A = fit_and_eval(X_A_train, X_A_test, "A: raw 182")
rf_B, yp_B, ypr_B, f1_B, auc_B, prauc_B = fit_and_eval(X_B_train, X_B_test, "B: raw + 103 traj")
rf_C, yp_C, ypr_C, f1_C, auc_C, prauc_C = fit_and_eval(X_C_train, X_C_test, "C: raw + traj + L1")

print("\n=== Classification reports ===")
for name, yp in [("A", yp_A), ("B", yp_B), ("C", yp_C)]:
    print(f"\n--- {name} ---")
    print(classification_report(y_test, yp, target_names=["licit","illicit"],
                                digits=4, zero_division=0))


train (t<= 34): n=29,894  illicit_rate=0.1158
test  (t >  34): n=16,670  illicit_rate=0.0650

Feature dims:
  A (raw):              182
  B (raw + traj):       285
  C (raw + traj + L1):  292

=== Training RFs ===
  [A: raw 182               ]  F1=0.8044  AUC=0.9269  PR-AUC=0.7927  (2.0s)
  [B: raw + 103 traj        ]  F1=0.8098  AUC=0.9333  PR-AUC=0.8097  (1.9s)
  [C: raw + traj + L1       ]  F1=0.8094  AUC=0.9361  PR-AUC=0.8093  (2.1s)

=== Classification reports ===

--- A ---
              precision    recall  f1-score   support

       licit     0.9781    0.9995    0.9887     15587
     illicit     0.9892    0.6777    0.8044      1083

    accuracy                         0.9786     16670
   macro avg     0.9837    0.8386    0.8965     16670
weighted avg     0.9788    0.9786    0.9767     16670


--- B ---
              precision    recall  f1-score   support

       licit     0.9786    0.9994    0.9889     15587
     illicit     0.9880    0.6861    0.8098      1083

    accuracy 

In [18]:
# ── Per-timestep test F1 across A / B / C ────────────────────────
print("=" * 70)
print("Per-timestep test F1 (illicit class)")
print("=" * 70)
print(f"{'t':>3}  {'n':>5}  {'illicit':>7}  {'A: raw':>8}  {'B: +traj':>8}  {'C: +L1':>8}  {'C-B':>7}  {'C-A':>7}")
for t in range(TRAIN_END + 1, N_TIME_STEPS + 1):
    m = (test_t_arr == t)
    if not m.any(): continue
    yt = y_test[m]
    if yt.sum() == 0:
        print(f"{t:>3}  {int(m.sum()):>5}  {int(yt.sum()):>7}  "
              f"{'NaN':>8}  {'NaN':>8}  {'NaN':>8}  {'NaN':>7}  {'NaN':>7}")
        continue
    f1_a = f1_score(yt, yp_A[m], pos_label=1, zero_division=0)
    f1_b = f1_score(yt, yp_B[m], pos_label=1, zero_division=0)
    f1_c = f1_score(yt, yp_C[m], pos_label=1, zero_division=0)
    print(f"{t:>3}  {int(m.sum()):>5}  {int(yt.sum()):>7}  "
          f"{f1_a:>8.4f}  {f1_b:>8.4f}  {f1_c:>8.4f}  "
          f"{f1_c-f1_b:>+7.3f}  {f1_c-f1_a:>+7.3f}")

# ── Top-30 feature importance from C, with feature-source tags ────
print("\n" + "=" * 70)
print("Top-30 features in model C, with source tags")
print("=" * 70)
all_feat_names = list(tx_feat_cols) + traj_cols + L1_cols
imp = rf_C.feature_importances_
order = np.argsort(-imp)[:30]
n_l1_top, n_traj_top = 0, 0
for rank, i in enumerate(order, 1):
    if i < F_TX:
        tag = ""
    elif i < F_TX + F_TRAJ:
        tag = "  (TRAJ)"; n_traj_top += 1
    else:
        tag = "  (L1-bipartite)"; n_l1_top += 1
    print(f"  {rank:2d}.  {imp[i]:.4f}  {all_feat_names[i]}{tag}")
print(f"\n  trajectory features in top-30: {n_traj_top} / 30")
print(f"  L1 bipartite features in top-30: {n_l1_top} / 30  (out of {F_L1} total L1 features)")

# ── Summary ──────────────────────────────────────────────────────
print("\n" + "=" * 70)
print(f"{'Model':30s}  {'F1':>8s}  {'AUC':>8s}  {'PR-AUC':>8s}")
print("=" * 70)
results = {
    "A: RF[raw 182]":            (f1_A, auc_A, prauc_A),
    "B: RF[raw + 103 traj]":     (f1_B, auc_B, prauc_B),
    "C: RF[raw + traj + 7 L1]":  (f1_C, auc_C, prauc_C),
}
for name, (f1, auc, prauc) in sorted(results.items(), key=lambda kv: -kv[1][0]):
    print(f"  {name:28s}  {f1:>8.4f}  {auc:>8.4f}  {prauc:>8.4f}")
print()
print(f"Δ C vs B (does L1 help on top of trajectory?):  F1={f1_C-f1_B:+.4f}  "
      f"AUC={auc_C-auc_B:+.4f}  PR-AUC={prauc_C-prauc_B:+.4f}")
print(f"Δ C vs A (full system vs raw RF):               F1={f1_C-f1_A:+.4f}  "
      f"AUC={auc_C-auc_A:+.4f}  PR-AUC={prauc_C-prauc_A:+.4f}")

if (f1_C - f1_B) >= 0.005:
    print("\n  ✅ Bipartite structural features add measurable signal beyond trajectory features.")
elif (f1_C - f1_B) >= 0.001 or (prauc_C - prauc_B) >= 0.005:
    print("\n  ⚠️  Marginal lift — bipartite structure helps on AUC/PR-AUC but not much on F1@0.5.")
else:
    print("\n  ❌ No meaningful lift from bipartite structural features over trajectory features.")
    print("     Possible reasons: PageRank captures structure already implicit in trajectory")
    print("     aggregates (count, recency, fan_asymmetry); cumulative bipartite structure")
    print("     may not add information beyond the per-wallet history features.")

print("\nNotes:")
print(f"  - Same train/test split as phase 1: train t<= {TRAIN_END}, test t > {TRAIN_END}.")
print(f"  - Same causality contract (strict τ < t for trajectory features; tx.t' <= t for L1).")
print(f"  - Layer 1 PageRank: undirected bipartite, damping={PR_DAMP}, {PR_ITERATIONS} power iterations.")
print(f"  - All three models: RandomForest(n_estimators=200, class_weight='balanced').")


Per-timestep test F1 (illicit class)
  t      n  illicit    A: raw  B: +traj    C: +L1      C-B      C-A
 35   1341      182    0.9805    0.9775    0.9805   +0.003   +0.000
 36   1708       33    0.8814    0.9000    0.9180   +0.018   +0.037
 37    498       40    0.7500    0.7692    0.7500   -0.019   +0.000
 38    756      111    0.9429    0.9479    0.9429   -0.005   +0.000
 39   1183       81    0.9128    0.9054    0.9272   +0.022   +0.014
 40   1211      112    0.7416    0.7486    0.7416   -0.007   +0.000
 41   1132      116    0.8900    0.9005    0.9108   +0.010   +0.021
 42   2154      239    0.8565    0.8599    0.8434   -0.016   -0.013
 43   1370       24    0.0000    0.0000    0.0000   +0.000   +0.000
 44   1591       24    0.0690    0.0667    0.0667   +0.000   -0.002
 45   1221        5    0.0000    0.0000    0.0000   +0.000   +0.000
 46    712        2    0.5000    0.5000    0.5000   +0.000   +0.000
 47    846       22    0.0000    0.0000    0.0000   +0.000   +0.000
 48    471 